# Shareability Metric

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
from ssl_shareability_metric.ssl_encoder_shareability import ssl_shareability
import torch
from torch.utils.data import TensorDataset, DataLoader
from ssl_shareability_metric.shared_encoder import SharedEncoder
from ssl_shareability_metric.whiten_and_center import whiten_and_center
import copy
from ssl_shareability_metric.seperate_encoder import SeperateEncoder

In [ ]:
df = pd.read_csv(Path().cwd().resolve().parent / "data/sample_data.csv")

## Collapse seperate date times into 1 date time feature

In [ ]:
df["datetime"] = pd.to_datetime(df[["year", "month", "day", "hour"]])

## Sort by date time to ensure t and t+1 relationship

In [ ]:
df = df.sort_values("datetime", ascending=True).reset_index(drop=True)

## Drop non continous features to preserve a clean cross covariance matrix M

In [ ]:
df = df.drop(columns=["No", "year", "month", "day", "hour", "wd", "station", "RAIN"])

## Handle NaN values

In [ ]:
df = df.dropna().reset_index(drop=True)

## Create lagged pairs

In [ ]:
current = df.copy()
future = df.shift(-1)

valid_pair = (df["datetime"].shift(-1) - df["datetime"]).eq(pd.Timedelta(hours=1))

x_current = current.loc[valid_pair].reset_index(drop=True)
x_future = future.loc[valid_pair].reset_index(drop=True)

x_current = x_current.drop(columns=["datetime"])
x_future = x_future.drop(columns=["datetime"])

x_current.head()

,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,WSPM
0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,4.4
1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,4.7
2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,5.6
3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,3.1
4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,2.0


## Convert to numpy for easier tensor work

In [ ]:
current_np = np.array(x_current)
future_np = np.array(x_future)

print(current_np.shape)
print(future_np.shape)

(30943, 10)
(30943, 10)


## Establish train val and test splits

In [ ]:
train_pct = 0.70
val_pct = 0.10

train_current = current_np[:int(train_pct * current_np.shape[0])]
train_future = future_np[:int(train_pct * future_np.shape[0])]

val_current = current_np[int(train_pct * current_np.shape[0]):int(train_pct * current_np.shape[0]) + int(val_pct * current_np.shape[0])]
val_future = future_np[int(train_pct * future_np.shape[0]):int(train_pct * future_np.shape[0]) + int(val_pct * future_np.shape[0])]

test_current = current_np[int(train_pct * current_np.shape[0]) + int(val_pct * current_np.shape[0]):]
test_future = future_np[int(train_pct * current_np.shape[0]) + int(val_pct * current_np.shape[0]):]

print(train_current.shape)
print(train_future.shape)

print(val_current.shape)
print(val_future.shape)

print(test_current.shape)
print(test_future.shape)

(21660, 10)
(21660, 10)
(3094, 10)
(3094, 10)
(6189, 10)
(6189, 10)


## Shareability score

In [ ]:
shareability, shared, seperate = ssl_shareability(train_current, train_future)
print(f"Shareability Score: {shareability}, Shared: {shared}, Seperate: {seperate}")

Shareability Score: 0.9999427343409215, Shared: 0.998177335180484, Seperate: 0.9982344997370264


## Whiten and center

In [ ]:
train_whitend_and_centered_current = whiten_and_center(train_current)
train_whitend_and_centered_future = whiten_and_center(train_future)
val_whitend_and_centered_current = whiten_and_center(val_current)
val_whitend_and_centered_future = whiten_and_center(val_future)
test_whitend_and_centered_current = whiten_and_center(test_current)
test_whitend_and_centered_future = whiten_and_center(test_future)

## Convert to tensors

In [ ]:
train_current_tensor = torch.tensor(train_whitend_and_centered_current, dtype=torch.float32)
train_future_tensor = torch.tensor(train_whitend_and_centered_future, dtype=torch.float32)
val_current_tensor = torch.tensor(val_whitend_and_centered_current, dtype=torch.float32)
val_future_tensor = torch.tensor(val_whitend_and_centered_future, dtype=torch.float32)
test_current_tensor = torch.tensor(test_whitend_and_centered_current, dtype=torch.float32)
test_future_tensor = torch.tensor(test_whitend_and_centered_future, dtype=torch.float32)

vector_size = train_current_tensor.shape[1]

train_pairs = TensorDataset(train_current_tensor, train_future_tensor)
val_pairs = TensorDataset(val_current_tensor, val_future_tensor)
test_pairs = TensorDataset(test_current_tensor, test_future_tensor)

train_dataloader = DataLoader(train_pairs, batch_size=64, shuffle=True)
val_dataloader = DataLoader(val_pairs, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_pairs, batch_size=64, shuffle=True)


## Shared encoder

In [ ]:
shared_encoder = SharedEncoder(vector_size=vector_size)
optimizer = torch.optim.Adam(shared_encoder.parameters(), lr=1e-2)


patience = 5
epochs = 10
patience_counter = 0
best_val_loss = float("inf")
delta = 1e-4
best_state = None

for i in range(epochs):
    train_loss = 0
    shared_encoder.train()
    for x, y in train_dataloader:
        Z_x = shared_encoder(x)
        Z_y = shared_encoder(y)
        loss = -torch.abs(torch.mean(Z_x * Z_y))
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            weights = shared_encoder.shared.weight
            weights.div_(weights.norm(p=2)) # divide in place by eucildean norm
             
    val_loss = 0
    shared_encoder.eval()
    for x, y in val_dataloader:
        with torch.no_grad():
            Z_x = shared_encoder(x)
            Z_y = shared_encoder(y)
            loss = -torch.abs(torch.mean(Z_x * Z_y))
            val_loss += loss.item()
            
    print(f"epoch: {i+1} train: {train_loss / len(train_dataloader)} val: {val_loss / len(val_dataloader)}")   
            
    avg_val_loss = val_loss / len(val_dataloader)
            
    if (avg_val_loss + delta) < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        best_state = copy.deepcopy(shared_encoder.state_dict())
    else:
        patience_counter += 1
        
    if patience_counter >= patience:
        break
        

epoch: 1 train: -0.8906644820991167 val: -0.9217466937035931
epoch: 2 train: -0.9038796295634414 val: -0.9319558301750495
epoch: 3 train: -0.9103061984949758 val: -0.9339818504391885
epoch: 4 train: -0.9124361769815462 val: -0.945606858146434
epoch: 5 train: -0.9141559475994392 val: -0.9378684114436714
epoch: 6 train: -0.9135298529381597 val: -0.9351005079794903
epoch: 7 train: -0.9110606050772653 val: -0.9407955201304689
epoch: 8 train: -0.9092683660245575 val: -0.9360279708492513
epoch: 9 train: -0.9090452646083889 val: -0.931466360481418
epoch: 10 train: -0.9086568979738736 val: -0.9379802376640086
epoch: 11 train: -0.9083180322056323 val: -0.9347953601759307
epoch: 12 train: -0.9086482553882936 val: -0.9277499427600783
epoch: 13 train: -0.9081674981257909 val: -0.932265077318464
epoch: 14 train: -0.907622262527809 val: -0.9301298613450966


## Seperate encoders

In [ ]:
seperate_encoder = SeperateEncoder(vector_size=vector_size)
optimizer = torch.optim.Adam(shared_encoder.parameters(), lr=1e-2)


patience = 5
epochs = 10
patience_counter = 0
best_val_loss = float("inf")
delta = 1e-4
best_state = None

for i in range(epochs):
    train_loss = 0
    shared_encoder.train()
    for x, y in train_dataloader:
        Z_x = shared_encoder(x)
        Z_y = shared_encoder(y)
        loss = -torch.abs(torch.mean(Z_x * Z_y))
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            weights = shared_encoder.shared.weight
            weights.div_(weights.norm(p=2)) # divide in place by eucildean norm
             
    val_loss = 0
    shared_encoder.eval()
    for x, y in val_dataloader:
        with torch.no_grad():
            Z_x = shared_encoder(x)
            Z_y = shared_encoder(y)
            loss = -torch.abs(torch.mean(Z_x * Z_y))
            val_loss += loss.item()
            
    print(f"epoch: {i+1} train: {train_loss / len(train_dataloader)} val: {val_loss / len(val_dataloader)}")   
            
    avg_val_loss = val_loss / len(val_dataloader)
            
    if (avg_val_loss + delta) < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        best_state = copy.deepcopy(shared_encoder.state_dict())
    else:
        patience_counter += 1
        
    if patience_counter >= patience:
        break
        